# CatBoost: Predicting Transit Time (Track A)

Predicts `transit_days` for each shipment. The goal from the project overview is to beat the naive "predict the promised date" baseline (MAE about 0.84 days) by a meaningful margin, and to hold up across modes and lanes.

- **Data:** `engineered_features.csv`, produced by `clean_data.ipynb`. Run that notebook first.
- **Metrics:** MAE (primary), RMSE, and the share of predictions within ±1 day.
- `test.csv` is held out for the finished model and is not used here. The test slice below is carved out of `train.csv`.

In [1]:
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import train_test_split

SEED = 42

df = pd.read_csv("engineered_features.csv", dtype={"origin_zip": str, "dest_zip": str})
assert df["shipment_id"].is_unique
df.shape

(39747, 61)

## 1. Train / validation / test split

Random 70 / 15 / 15 split. Validation is used for every modelling choice. The test slice is scored once, at the very end. Other models should use the same `SEED` and proportions so results are directly comparable.

In [2]:
train, holdout = train_test_split(df, test_size=0.30, random_state=SEED)
val, test = train_test_split(holdout, test_size=0.50, random_state=SEED)

assert len(train) + len(val) + len(test) == len(df)
len(train), len(val), len(test)

(27822, 5962, 5963)

## 2. Baselines

- **Promised date:** predict `promised_days`. This is the bar set in the overview.
- **Lookup table:** the median transit time per carrier × service level × zone, learned from the training rows. A stronger test: if the model can't clearly beat a lookup table, it isn't adding much.

In [3]:
TARGET = "transit_days"
LOOKUP_KEYS = ["carrier", "service_level", "zone"]

def score(y_true, y_pred):
    errors = np.asarray(y_true) - np.asarray(y_pred)
    return {
        "MAE": np.abs(errors).mean(),
        "RMSE": np.sqrt((errors ** 2).mean()),
        "within_1_day_pct": 100 * (np.abs(errors) <= 1).mean(),
    }

lookup = train.groupby(LOOKUP_KEYS)[TARGET].median().rename("lookup_pred").reset_index()

def predict_lookup(data):
    preds = data[LOOKUP_KEYS].merge(lookup, on=LOOKUP_KEYS, how="left")["lookup_pred"]
    return preds.fillna(train[TARGET].median()).values

pd.DataFrame({
    "promised_date": score(val[TARGET], val["promised_days"]),
    "lookup_table": score(val[TARGET], predict_lookup(val)),
}).T.round(4)

,MAE,RMSE,within_1_day_pct
promised_date,0.8345,1.1461,84.1496
lookup_table,0.6094,0.9370,93.1567


## 3. Features

IDs, raw dates, ZIPs and raw coordinates are left out: the metro and distance columns already carry that information. `delivered_on_time` is excluded because it's derived from the target, so using it would be leakage.

In [4]:
BASE_CATS = ["shipment_mode", "carrier", "service_level", "origin_metro", "dest_metro"]
BASE_NUMS = ["ship_dow", "ship_month", "distance_miles", "zone",
             "weight_lbs", "length_in", "width_in", "height_in", "pallet_count",
             "residential_flag", "signature_required", "declared_value_usd",
             "weight_lbs_na", "length_in_na", "width_in_na", "height_in_na",
             "fuel_surcharge_pct", "weather_severity", "carrier_congestion_index",
             "promised_days", "carrier_cost_usd"]

ENGINEERED_CATS = ["lane", "carrier_service"]
ENGINEERED_NUMS = ["order_to_ship_days", "ship_week", "ship_quarter", "is_weekend_ship", "is_peak_season", "is_winter",
                   "lat_delta", "lon_delta",
                   "volume_cuft", "dim_weight_lbs", "billable_weight_lbs", "density_lb_per_cuft", "weight_per_pallet",
                   "any_dim_missing", "has_declared_value", "log_declared_value",
                   "peak_congestion", "weather_x_congestion", "weather_x_distance", "promised_crosses_weekend",
                   "cost_per_billable_lb", "cost_per_mile"]

FEATURE_SETS = {
    "original columns": (BASE_CATS, BASE_NUMS),
    "original + engineered": (BASE_CATS + ENGINEERED_CATS, BASE_NUMS + ENGINEERED_NUMS),
}

excluded = {TARGET, "delivered_on_time"}
for cats, nums in FEATURE_SETS.values():
    assert not excluded & set(cats + nums), "target leaked into features"
    assert set(cats + nums) <= set(df.columns)

## 4. Compare feature sets and targets

The loss is MAE because that's what the project is scored on. Early stopping on validation picks the number of trees.

Two questions, each answered on validation:
- Do the engineered features help?
- Is it better to predict `transit_days` directly, or the delay (`transit_days - promised_days`) and add the promise back?

In [5]:
def fit_catboost(cats, nums, predict_delay=False, **params):
    features = cats + nums

    def target(data):
        return data[TARGET] - data["promised_days"] if predict_delay else data[TARGET]

    settings = dict(iterations=1500, learning_rate=0.08, depth=6, loss_function="MAE", eval_metric="MAE",
                    early_stopping_rounds=100, random_seed=SEED, verbose=0, allow_writing_files=False)
    settings.update(params)
    model = CatBoostRegressor(**settings)
    model.fit(Pool(train[features], target(train), cat_features=cats),
              eval_set=Pool(val[features], target(val), cat_features=cats))
    return model

def predict_days(model, data, cats, nums, predict_delay=False):
    preds = model.predict(data[cats + nums])
    return preds + data["promised_days"].values if predict_delay else preds

rows = []
for set_name, (cats, nums) in FEATURE_SETS.items():
    for predict_delay in [False, True]:
        model = fit_catboost(cats, nums, predict_delay)
        preds = predict_days(model, val, cats, nums, predict_delay)
        rows.append({"features": set_name, "target": "delay" if predict_delay else "transit_days",
                     **score(val[TARGET], preds), "trees": model.tree_count_})

comparison = pd.DataFrame(rows).sort_values("MAE").reset_index(drop=True)
comparison.round(4)

,features,target,MAE,RMSE,within_1_day_pct,trees
0,original columns,transit_days,0.5317,0.7679,87.8061,390
1,original + engineered,delay,0.5324,0.7703,87.5713,692
2,original columns,delay,0.5330,0.7688,87.6048,354
3,original + engineered,transit_days,0.5356,0.7700,87.7726,390


## 5. Light tuning

A small grid on the best setup from section 4. It's kept small so validation isn't overfit by trying too many options.

In [6]:
best_set = comparison.loc[0, "features"]
best_delay = comparison.loc[0, "target"] == "delay"
best_cats, best_nums = FEATURE_SETS[best_set]
print("tuning:", best_set, "| target:", comparison.loc[0, "target"])

rows = []
for depth in [4, 6, 8]:
    for learning_rate in [0.03, 0.08]:
        model = fit_catboost(best_cats, best_nums, best_delay, depth=depth, learning_rate=learning_rate)
        preds = predict_days(model, val, best_cats, best_nums, best_delay)
        rows.append({"depth": depth, "learning_rate": learning_rate, **score(val[TARGET], preds), "trees": model.tree_count_})

tuning = pd.DataFrame(rows).sort_values("MAE").reset_index(drop=True)
tuning.round(4)

tuning: original columns | target: transit_days


,depth,learning_rate,MAE,RMSE,within_1_day_pct,trees
0,6,0.03,0.5298,0.7657,87.8397,857
1,8,0.03,0.5306,0.7683,87.7390,572
2,4,0.03,0.5307,0.7648,87.8900,1487
3,4,0.08,0.5314,0.7657,87.8732,856
4,6,0.08,0.5317,0.7679,87.8061,390
5,8,0.08,0.5332,0.7705,87.5042,219


## 6. Final model and rounding

`transit_days` is always a whole number, so this checks on validation whether rounding the prediction helps.

In [7]:
final_model = fit_catboost(best_cats, best_nums, best_delay,
                           depth=int(tuning.loc[0, "depth"]), learning_rate=float(tuning.loc[0, "learning_rate"]))

def predict_final(data, rounded):
    preds = predict_days(final_model, data, best_cats, best_nums, best_delay)
    return np.round(preds).clip(min=1) if rounded else preds

pd.DataFrame({
    "unrounded": score(val[TARGET], predict_final(val, rounded=False)),
    "rounded": score(val[TARGET], predict_final(val, rounded=True)),
}).T.round(4)

,MAE,RMSE,within_1_day_pct
unrounded,0.5298,0.7657,87.8397
rounded,0.4710,0.8170,96.6622


## 7. Final score on the test slice

Every choice above was made on validation, so this is the honest estimate of performance. Don't go back and change the model based on this table, or it stops being one.

In [8]:
test_results = pd.DataFrame({
    "promised_date": score(test[TARGET], test["promised_days"]),
    "lookup_table": score(test[TARGET], predict_lookup(test)),
    "catboost": score(test[TARGET], predict_final(test, rounded=False)),
    "catboost_rounded": score(test[TARGET], predict_final(test, rounded=True)),
}).T
test_results.round(4)

,MAE,RMSE,within_1_day_pct
promised_date,0.8419,1.1535,84.7895
lookup_table,0.6251,0.9493,92.9733
catboost,0.5487,0.7882,87.5901
catboost_rounded,0.4977,0.8416,96.1597


## 8. Error analysis

Where is the model weakest, and does it beat the promised date everywhere? All tables use unrounded predictions on the test slice.

In [9]:
errors = test.assign(
    model_error=np.abs(test[TARGET] - predict_final(test, rounded=False)),
    promised_error=np.abs(test[TARGET] - test["promised_days"]),
)

def error_by(col):
    table = errors.groupby(col).agg(
        shipments=("model_error", "size"),
        model_MAE=("model_error", "mean"),
        promised_MAE=("promised_error", "mean"),
        within_1_day_pct=("model_error", lambda e: 100 * (e <= 1).mean()),
    )
    return table.sort_values("model_MAE", ascending=False).round(3)

for col in ["shipment_mode", "carrier", "service_level", "zone", "ship_month"]:
    display(error_by(col))

,shipments,model_MAE,promised_MAE,within_1_day_pct
shipment_mode,,,,
parcel,5111,0.550,0.856,87.400
ltl_freight,852,0.539,0.756,88.732


,shipments,model_MAE,promised_MAE,within_1_day_pct
carrier,,,,
Falcon Post,2031,0.654,0.874,80.896
Ironclad Freight,506,0.581,0.731,84.980
Northstar Parcel,1709,0.524,0.688,88.473
Vela Freight,346,0.479,0.792,94.220
Cobalt Regional,186,0.457,0.839,96.237
Summit Express,1185,0.425,1.070,95.612


,shipments,model_MAE,promised_MAE,within_1_day_pct
service_level,,,,
Ground,2849,0.553,0.874,87.048
TwoDay,738,0.548,0.869,87.805
Expedited,1034,0.548,0.829,87.621
LTL Guaranteed,234,0.547,0.709,86.752
Overnight,490,0.541,0.792,88.367
LTL Standard,618,0.536,0.773,89.482


,shipments,model_MAE,promised_MAE,within_1_day_pct
zone,,,,
3,236,0.599,0.860,85.169
6,1154,0.579,0.831,87.002
7,976,0.561,0.864,85.246
2,27,0.560,0.667,92.593
8,1097,0.556,0.820,87.511
5,1771,0.531,0.874,88.312
4,702,0.499,0.782,90.741


,shipments,model_MAE,promised_MAE,within_1_day_pct
ship_month,,,,
1,528,0.592,0.862,87.689
10,495,0.590,0.893,85.253
9,492,0.564,0.823,87.398
6,523,0.558,0.837,87.572
4,511,0.548,0.881,87.280
2,459,0.546,0.795,86.710
12,499,0.545,0.872,87.976
3,498,0.538,0.865,87.751
8,493,0.533,0.811,89.047


In [10]:
# Lanes need enough test shipments for their MAE to mean anything
MIN_LANE_SHIPMENTS = 15

lanes = error_by("lane").query("shipments >= @MIN_LANE_SHIPMENTS")
print(f"{len(lanes)} lanes with at least {MIN_LANE_SHIPMENTS} test shipments")
print(f"lanes where the model beats the promised date: {(lanes['model_MAE'] < lanes['promised_MAE']).sum()}")
print(f"lane MAE range: {lanes['model_MAE'].min():.3f} to {lanes['model_MAE'].max():.3f}")
lanes.head(10)

200 lanes with at least 15 test shipments
lanes where the model beats the promised date: 186
lane MAE range: 0.302 to 1.460


,shipments,model_MAE,promised_MAE,within_1_day_pct
lane,,,,
Houston TX -> Portland OR,18,1.460,1.333,66.667
Denver CO -> Columbus OH,16,1.437,1.688,81.250
New York NY -> Kansas City MO,26,1.103,0.962,61.538
Phoenix AZ -> Kansas City MO,16,0.947,1.125,68.750
Miami FL -> Phoenix AZ,19,0.893,0.789,78.947
Seattle WA -> Charlotte NC,17,0.839,0.941,76.471
Atlanta GA -> Minneapolis MN,31,0.833,0.806,77.419
Atlanta GA -> Phoenix AZ,32,0.832,1.062,75.000
Columbus OH -> Los Angeles CA,19,0.806,1.000,73.684


In [11]:
pd.Series(final_model.get_feature_importance(), index=best_cats + best_nums, name="importance") \
    .sort_values(ascending=False).head(15).round(2)

promised_days               43.42
carrier                     17.46
service_level                9.79
zone                         6.05
distance_miles               5.28
weather_severity             5.09
ship_dow                     4.27
carrier_cost_usd             3.03
ship_month                   1.64
carrier_congestion_index     1.15
residential_flag             0.55
width_in                     0.47
fuel_surcharge_pct           0.32
height_in                    0.30
length_in                    0.25
Name: importance, dtype: float64

## 9. Results

Test slice, `SEED = 42`:

| Model | MAE | RMSE | Within ±1 day |
|---|---|---|---|
| Promised date | 0.842 | 1.154 | 84.8% |
| Lookup table | 0.625 | 0.949 | 93.0% |
| CatBoost | 0.549 | 0.788 | 87.6% |
| CatBoost, rounded | 0.498 | 0.842 | 96.2% |

- **It clears the bar:** rounded MAE is about 41% below the promised-date baseline. Rounding lowers MAE and raises the ±1 day share, but makes RMSE slightly worse.
- **Modes:** parcel and freight perform about the same (MAE 0.550 and 0.539).
- **Weakest carrier:** Falcon Post (MAE 0.654, 80.9% within ±1 day) has noticeably more unexplained variation than the others.
- **Lanes:** the model beats the promised date on 186 of the 200 lanes with at least 15 test shipments. Lane-level figures rest on few shipments, so treat the extremes with caution.
- **Engineered features didn't help:** the original columns scored 0.532 on validation, against 0.536 with engineered features added. The gap is small enough to be noise.
- **Predicting delay didn't help either:** it was no better than predicting `transit_days` directly.
- **Limitation:** this is a random split. A time-ordered split would be a tougher test of how the model handles future shipments.